# Supply Chain Demand Forecasting & Inventory Optimization

Dataset: [Kaggle — Store Item Demand Forecasting Challenge](https://www.kaggle.com/c/demand-forecasting-kernels-only/data)
913,000 rows | 10 stores x 50 items x 5 years (2013–2017)

**Before running:** make sure `train.csv` is inside a `data/raw/` folder next to this notebook.
Run every cell in order, top to bottom, using Shift+Enter.

## Step 0 — Setup: imports and folders

In [ ]:
# If you haven't installed the libraries yet, uncomment and run this once:
# !pip install pandas numpy lightgbm xgboost scikit-learn matplotlib

import os
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
import matplotlib.pyplot as plt
%matplotlib inline

# Make sure the folders we'll save outputs to actually exist
os.makedirs("data/processed", exist_ok=True)
os.makedirs("models", exist_ok=True)
os.makedirs("reports/figures", exist_ok=True)

print("Current working directory:", os.getcwd())
print("Looking for data at:", os.path.abspath("data/raw/train.csv"))

**If the path above doesn't point to where your `train.csv` actually is**, run this in the cell below
(edit the path to match your computer), then re-run the cell above to confirm:

```python
os.chdir(r"C:\Users\YourName\projects\demand-forecasting-inventory-opt")
```

## Step 1 — Load and inspect the data

We load `train.csv` and take a first look: how many rows, what date range, any missing values.

In [ ]:
train_raw = pd.read_csv("data/raw/train.csv", parse_dates=["date"])

print("Shape:", train_raw.shape)
print("Date range:", train_raw.date.min(), "to", train_raw.date.max())
print("Stores:", train_raw.store.nunique(), "| Items:", train_raw.item.nunique())
print("\nMissing values:\n", train_raw.isnull().sum())
train_raw.head()

## Step 2 — Quick exploratory look

A couple of simple plots just to eyeball the data before we model it: total sales over time, and sales by
store, to confirm the seasonality and store differences we're expecting.

In [ ]:
daily_total = train_raw.groupby("date")["sales"].sum()

plt.figure(figsize=(12, 4))
plt.plot(daily_total.index, daily_total.values)
plt.title("Total Daily Sales Across All Stores/Items")
plt.xlabel("Date")
plt.ylabel("Units Sold")
plt.show()

In [ ]:
store_avg = train_raw.groupby("store")["sales"].mean().sort_values(ascending=False)

plt.figure(figsize=(8, 4))
store_avg.plot(kind="bar", color="#4C72B0")
plt.title("Average Daily Sales per Store")
plt.xlabel("Store")
plt.ylabel("Avg Units Sold/Day")
plt.show()

## Step 3 — Feature engineering

This is the core of the project. We turn the raw `date, store, item, sales` columns into ~40 features the
model can actually learn from: calendar signals, seasonality, lag features, rolling statistics, exponentially
weighted averages, and group-level averages.

**Key rule:** every lag/rolling/EWM feature is built using `.shift(1)` first, so the model never sees
today's own sales value when predicting today. This avoids data leakage.

In [ ]:
def add_calendar_features(df):
    """Basic calendar features: 8 features."""
    df["dayofweek"] = df["date"].dt.dayofweek
    df["day"] = df["date"].dt.day
    df["month"] = df["date"].dt.month
    df["year"] = df["date"].dt.year
    df["weekofyear"] = df["date"].dt.isocalendar().week.astype(int)
    df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)
    df["is_month_start"] = df["date"].dt.is_month_start.astype(int)
    df["is_month_end"] = df["date"].dt.is_month_end.astype(int)
    return df


def add_fourier_seasonality(df):
    """Smooth cyclical encoding so Dec/Jan are treated as neighbours: 4 features."""
    day_of_year = df["date"].dt.dayofyear
    df["sin_year"] = np.sin(2 * np.pi * day_of_year / 365.25)
    df["cos_year"] = np.cos(2 * np.pi * day_of_year / 365.25)
    df["sin_week"] = np.sin(2 * np.pi * df["dayofweek"] / 7)
    df["cos_week"] = np.cos(2 * np.pi * df["dayofweek"] / 7)
    return df


def add_lag_features(df, lags=(1, 7, 14, 28, 90, 365)):
    """Sales N days ago, per store-item pair: 6 features."""
    grp = df.groupby(["store", "item"], sort=False)["sales"]
    for lag in lags:
        df[f"lag_{lag}"] = grp.shift(lag).astype("float32")
    return df


def add_rolling_features(df, windows=(7, 14, 30, 90)):
    """Rolling mean/std/min/max over trailing windows: 16 features.
    Computed on shifted sales so today's value is never included."""
    df = df.sort_values(["store", "item", "date"]).reset_index(drop=True)
    df["_shifted_sales"] = df.groupby(["store", "item"], sort=False)["sales"].shift(1)

    for w in windows:
        roll = (
            df.groupby(["store", "item"], sort=False)["_shifted_sales"]
            .rolling(window=w, min_periods=1)
            .agg(["mean", "std", "min", "max"])
            .reset_index(drop=True)
        )
        df[f"roll_mean_{w}"] = roll["mean"].astype("float32")
        df[f"roll_std_{w}"] = roll["std"].astype("float32")
        df[f"roll_min_{w}"] = roll["min"].astype("float32")
        df[f"roll_max_{w}"] = roll["max"].astype("float32")

    df.drop(columns=["_shifted_sales"], inplace=True)
    return df


def add_ewm_features(df, spans=(7, 30, 90)):
    """Exponentially weighted mean, more weight on recent days: 3 features."""
    df["_shifted_sales"] = df.groupby(["store", "item"], sort=False)["sales"].shift(1)
    for span in spans:
        df[f"ewm_{span}"] = (
            df.groupby(["store", "item"], sort=False)["_shifted_sales"]
            .transform(lambda s: s.ewm(span=span, min_periods=1).mean())
            .astype("float32")
        )
    df.drop(columns=["_shifted_sales"], inplace=True)
    return df


def add_group_aggregates(df, train_ref):
    """Historical average behaviour per store / item / pair: 3 features.
    Computed ONLY from train_ref to avoid leaking validation info."""
    store_mean = train_ref.groupby("store")["sales"].mean().rename("store_avg_sales")
    item_mean = train_ref.groupby("item")["sales"].mean().rename("item_avg_sales")
    pair_mean = train_ref.groupby(["store", "item"])["sales"].mean().rename("pair_avg_sales")
    df = df.merge(store_mean, on="store", how="left")
    df = df.merge(item_mean, on="item", how="left")
    df = df.merge(pair_mean, on=["store", "item"], how="left")
    return df


def build_features(df, train_ref):
    """Runs the full feature pipeline."""
    df = df.copy()
    df["store"] = df["store"].astype("int16")
    df["item"] = df["item"].astype("int16")
    df["sales"] = df["sales"].astype("float32")
    df = add_calendar_features(df)
    df = add_fourier_seasonality(df)
    df = add_lag_features(df)
    df = add_rolling_features(df)
    df = add_ewm_features(df)
    df = add_group_aggregates(df, train_ref)
    return df


FEATURE_COLUMNS = [
    "store", "item", "dayofweek", "day", "month", "year", "weekofyear",
    "is_weekend", "is_month_start", "is_month_end",
    "sin_year", "cos_year", "sin_week", "cos_week",
    "lag_1", "lag_7", "lag_14", "lag_28", "lag_90", "lag_365",
    "roll_mean_7", "roll_std_7", "roll_min_7", "roll_max_7",
    "roll_mean_14", "roll_std_14", "roll_min_14", "roll_max_14",
    "roll_mean_30", "roll_std_30", "roll_min_30", "roll_max_30",
    "roll_mean_90", "roll_std_90", "roll_min_90", "roll_max_90",
    "ewm_7", "ewm_30", "ewm_90",
    "store_avg_sales", "item_avg_sales", "pair_avg_sales",
]

print(f"{len(FEATURE_COLUMNS)} feature columns defined.")

## Step 4 — Time-based train/validation split

We hold out the **last 90 days** of 2017 as validation. We never shuffle randomly — time series data must
always be split so validation comes *after* training in time, otherwise the model effectively "sees the
future" and your score becomes meaningless.

In [ ]:
VAL_DAYS = 90
cutoff = train_raw["date"].max() - pd.Timedelta(days=VAL_DAYS)

train_df = train_raw[train_raw["date"] <= cutoff].copy()
val_df = train_raw[train_raw["date"] > cutoff].copy()

print(f"Train: {train_df.shape} | Validation: {val_df.shape}")
print(f"Train dates: {train_df.date.min()} to {train_df.date.max()}")
print(f"Val dates:   {val_df.date.min()} to {val_df.date.max()}")

In [ ]:
# Build features on the combined data so lag/rolling features for the
# validation period can look back correctly into the training period.
combined = pd.concat([train_df, val_df], axis=0).sort_values(["store", "item", "date"])
combined_feat = build_features(combined, train_ref=train_df)

train_feat = combined_feat[combined_feat["date"] <= train_df["date"].max()].copy()
val_feat = combined_feat[combined_feat["date"] > train_df["date"].max()].copy()

# Drop early rows where lag_365 can't be computed yet (first year of data)
train_feat = train_feat.dropna(subset=["lag_365"])

X_train, y_train = train_feat[FEATURE_COLUMNS], train_feat["sales"]
X_val, y_val = val_feat[FEATURE_COLUMNS], val_feat["sales"]

print(f"Final training rows after dropna: {len(X_train)}")
X_train.head()

## Step 5 — SMAPE metric

SMAPE (Symmetric Mean Absolute Percentage Error) is the standard metric for demand forecasting because it's
scale-independent — an error of 5 units matters a lot for a slow-selling item but very little for a
best-seller. Lower SMAPE = better.

In [ ]:
def smape(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    denom = np.abs(y_true) + np.abs(y_pred)
    denom = np.where(denom == 0, 1, denom)  # avoid divide-by-zero on zero-sales days
    return 100 * np.mean(2 * np.abs(y_pred - y_true) / denom)

## Step 6 — Naive seasonal baseline

Before training any ML model, we need something to beat. The simplest realistic approach: predict this
year's sales using **last year's sales for the same day**. Whatever SMAPE this scores becomes our bar.

In [ ]:
lookup = train_raw.set_index(["store", "item", "date"])["sales"]

baseline_preds = []
for _, row in val_df.iterrows():
    key = (row["store"], row["item"], row["date"] - pd.Timedelta(days=365))
    baseline_preds.append(lookup.get(key, np.nan))

baseline_preds = pd.Series(baseline_preds, index=val_df.index)
baseline_preds = baseline_preds.fillna(val_df["sales"].mean())

baseline_smape = smape(val_df["sales"].values, baseline_preds.values)
print(f"Naive baseline SMAPE: {baseline_smape:.3f}")

## Step 7 — Train LightGBM

LightGBM builds hundreds of small decision trees one after another, each correcting the mistakes of the
ones before it ("gradient boosting"). `early_stopping` automatically stops training once the validation
score stops improving, protecting against overfitting.

In [ ]:
lgb_train = lgb.Dataset(X_train, label=y_train)
lgb_val = lgb.Dataset(X_val, label=y_val, reference=lgb_train)

lgb_params = {
    "objective": "regression",
    "metric": "mae",
    "learning_rate": 0.05,
    "num_leaves": 64,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "verbose": -1,
}

lgb_model = lgb.train(
    lgb_params, lgb_train, num_boost_round=1000,
    valid_sets=[lgb_val],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],
)

lgb_preds = lgb_model.predict(X_val, num_iteration=lgb_model.best_iteration)
lgb_smape = smape(y_val.values, lgb_preds)
print(f"\nLightGBM SMAPE: {lgb_smape:.3f}")

## Step 8 — Train XGBoost

Same idea, a different library. XGBoost and LightGBM build trees slightly differently, so they tend to make different errors — useful for the ensemble step next.

In [ ]:
xgb_train = xgb.DMatrix(X_train, label=y_train)
xgb_val = xgb.DMatrix(X_val, label=y_val)

xgb_params = {
    "objective": "reg:squarederror",
    "eta": 0.05,
    "max_depth": 8,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "eval_metric": "mae",
}

xgb_model = xgb.train(
    xgb_params, xgb_train, num_boost_round=1000,
    evals=[(xgb_val, "val")],
    early_stopping_rounds=50, verbose_eval=100,
)

xgb_preds = xgb_model.predict(xgb_val, iteration_range=(0, xgb_model.best_iteration))
xgb_smape = smape(y_val.values, xgb_preds)
print(f"\nXGBoost SMAPE: {xgb_smape:.3f}")

## Step 9 — Ensemble the two models

Rather than assuming a 50/50 blend, we search every weight from 0% to 100% in 5% steps and keep whichever
scores best on validation SMAPE.

In [ ]:
best_w, best_smape = 0.5, 999
for w in np.arange(0, 1.05, 0.05):
    blend = w * lgb_preds + (1 - w) * xgb_preds
    s = smape(y_val.values, blend)
    if s < best_smape:
        best_smape, best_w = s, w

ensemble_preds = best_w * lgb_preds + (1 - best_w) * xgb_preds
print(f"Best ensemble weight (LightGBM share): {best_w:.2f}")
print(f"Ensemble SMAPE: {best_smape:.3f}")

## Step 10 — Results summary

In [ ]:
improvement = 100 * (baseline_smape - best_smape) / baseline_smape

results = pd.DataFrame({
    "Model": ["Naive Baseline", "LightGBM", "XGBoost", "Ensemble"],
    "SMAPE": [baseline_smape, lgb_smape, xgb_smape, best_smape],
})
print(results.to_string(index=False))
print(f"\nImprovement vs baseline: {improvement:.1f}%")

In [ ]:
plt.figure(figsize=(7, 4.5))
bars = plt.bar(results["Model"], results["SMAPE"],
               color=["#999999", "#4C72B0", "#DD8452", "#55A868"])
plt.ylabel("SMAPE (lower is better)")
plt.title("Forecast Accuracy: Baseline vs ML Models")
for bar, val in zip(bars, results["SMAPE"]):
    plt.text(bar.get_x() + bar.get_width()/2, val + 0.3, f"{val:.2f}", ha="center")
plt.tight_layout()
plt.savefig("reports/figures/smape_comparison.png", dpi=150)
plt.show()

### Actual vs predicted for one sample store-item pair

In [ ]:
val_feat = val_feat.copy()
val_feat["y_true"] = y_val.values
val_feat["y_pred_ensemble"] = ensemble_preds

sample = val_feat[(val_feat["store"] == 1) & (val_feat["item"] == 1)].sort_values("date")

plt.figure(figsize=(10, 4.5))
plt.plot(sample["date"], sample["y_true"], label="Actual", linewidth=2)
plt.plot(sample["date"], sample["y_pred_ensemble"], label="Predicted (Ensemble)", linewidth=2, linestyle="--")
plt.title("Actual vs Predicted Sales — Store 1, Item 1")
plt.xlabel("Date")
plt.ylabel("Units Sold")
plt.legend()
plt.tight_layout()
plt.savefig("reports/figures/actual_vs_predicted.png", dpi=150)
plt.show()

### Feature importance — which of our 40 features mattered most

In [ ]:
importance = pd.DataFrame({
    "feature": lgb_model.feature_name(),
    "importance": lgb_model.feature_importance(importance_type="gain"),
}).sort_values("importance", ascending=True).tail(15)

plt.figure(figsize=(8, 6))
plt.barh(importance["feature"], importance["importance"], color="#4C72B0")
plt.title("Top 15 Feature Importances (LightGBM, gain)")
plt.tight_layout()
plt.savefig("reports/figures/feature_importance.png", dpi=150)
plt.show()

## Step 11 — Save models and predictions

So we don't have to retrain every time, and so the inventory step below can reuse these predictions.

In [ ]:
lgb_model.save_model("models/lgbm_model.txt")
xgb_model.save_model("models/xgb_model.json")
val_feat.to_csv("data/processed/val_predictions.csv", index=False)
print("Saved models to models/ and predictions to data/processed/val_predictions.csv")

## Step 12 — Inventory optimization

Now we turn forecasts into actual replenishment decisions per store-item pair: **safety stock**, **reorder
point**, and **EOQ (Economic Order Quantity)** at a 95% service level.

**Assumptions** (the raw dataset has no cost/lead-time data, so these are declared placeholders — document
them clearly in your README):
- Lead time: 7 days
- Ordering cost: $50/order
- Holding cost: 20% of unit value/year (unit value assumed $1 as a stand-in)
- Service level: 95% → Z-score = 1.645

In [ ]:
Z_95 = 1.645
LEAD_TIME_DAYS = 7
ORDERING_COST = 50.0
HOLDING_COST_PER_UNIT_YEAR = 0.20

# Step A: how wrong is our forecast, per store-item pair? This becomes our
# estimate of demand uncertainty (sigma) used in the safety stock formula.
val_feat["error"] = val_feat["y_true"] - val_feat["y_pred_ensemble"]

demand_stats = (
    val_feat.groupby(["store", "item"])
    .agg(forecast_error_std=("error", "std"),
         avg_daily_demand=("y_pred_ensemble", "mean"))
    .reset_index()
)
demand_stats["forecast_error_std"] = demand_stats["forecast_error_std"].fillna(
    demand_stats["forecast_error_std"].mean()
)

demand_stats.head()

In [ ]:
policy = demand_stats.copy()

# Safety Stock: buffer against demand variability during lead time
policy["safety_stock"] = Z_95 * policy["forecast_error_std"] * np.sqrt(LEAD_TIME_DAYS)

# Reorder Point: stock level that triggers a new purchase order
policy["reorder_point"] = (policy["avg_daily_demand"] * LEAD_TIME_DAYS) + policy["safety_stock"]

# EOQ: cost-optimal order quantity
policy["annual_demand"] = policy["avg_daily_demand"] * 365
policy["eoq"] = np.sqrt(
    (2 * policy["annual_demand"] * ORDERING_COST) / HOLDING_COST_PER_UNIT_YEAR
)

for col in ["safety_stock", "reorder_point", "eoq"]:
    policy[col] = policy[col].round().astype(int)

policy = policy[["store", "item", "avg_daily_demand", "forecast_error_std",
                  "safety_stock", "reorder_point", "eoq"]].sort_values(["store", "item"])

policy.to_csv("data/processed/inventory_policy.csv", index=False)
print(f"Computed inventory policy for {len(policy)} store-item pairs.")
policy.head(10)

## Done

You now have:
- `models/lgbm_model.txt`, `models/xgb_model.json` — trained models
- `data/processed/val_predictions.csv` — validation predictions
- `data/processed/inventory_policy.csv` — per-SKU replenishment table (500 rows)
- `reports/figures/*.png` — the three charts, ready to embed in your GitHub README

Next: push this project folder to GitHub. See the README.md in your project root for the exact commands.